# CricShield C2 – Delivery Event Detection (Colab)
Run the cells top to bottom. Your data stays in Google Drive, so nothing is lost when Colab disconnects.

**Before you start:** upload `cricshield_event_detection.zip` to **MyDrive/CricShield_C2/** and your bowling videos to **MyDrive/CricShield_C2/data/videos/** (one delivery per clip).

## 1. Connect Google Drive and set up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/CricShield_C2'
os.makedirs(f'{ROOT}/data/videos', exist_ok=True)
os.makedirs(f'{ROOT}/models', exist_ok=True)
%cd {ROOT}
!unzip -oq cricshield_event_detection.zip
%cd {ROOT}/cricshield_event_detection
!pip install -q mediapipe xgboost

## 2. Download the MediaPipe pose model (once)

In [ ]:
MODEL = f'{ROOT}/models/pose_landmarker_heavy.task'
if not os.path.exists(MODEL):
    !wget -q -O "{MODEL}" https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task
print('model ready:', os.path.getsize(MODEL) // 1_000_000, 'MB')

## 3. Check the setup with FAKE data (never report these numbers)

In [ ]:
!python make_synthetic.py --out /content/demo_data
!python train_evaluate.py --landmarks /content/demo_data/landmarks --labels /content/demo_data/labels.csv --out /content/demo_results

## 4. Extract landmarks from your videos

In [ ]:
!python extract_landmarks.py --videos "{ROOT}/data/videos" --out "{ROOT}/data/landmarks" --model "{MODEL}"

## 5. Label BFC, FFC and BR (Colab version)
Colab can't open the pop-up labelling window, so instead:
1. Run `show_frames(...)` to see a grid of numbered frames.
2. Zoom in with a smaller range and `step=1` until you find the exact frame.
3. Save the three frame numbers with `add_label(...)`.

Rules: **contact = first frame the foot touches the ground; release = last frame the ball is still touching the fingers.**

In [ ]:
import cv2, math, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

VIDEOS = Path(ROOT) / 'data/videos'
LABELS = Path(ROOT) / 'data/labels.csv'

def video_info(name):
    cap = cv2.VideoCapture(str(VIDEOS / name))
    n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f'{name}: {n} frames at {fps:.1f} fps')
    return n, fps

def show_frames(name, start=0, end=None, step=5, cols=6, crop_bottom=None):
    """Grid of frames start..end. crop_bottom=0.5 shows only the lower half (feet)."""
    cap = cv2.VideoCapture(str(VIDEOS / name))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = min(end if end is not None else n - 1, n - 1)
    idx = list(range(start, end + 1, step))
    rows = math.ceil(len(idx) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows))
    axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]
    for ax in axes: ax.axis('off')
    for ax, i in zip(axes, idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, f = cap.read()
        if not ok: continue
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        if crop_bottom: f = f[int(f.shape[0] * (1 - crop_bottom)):]
        ax.imshow(f); ax.set_title(f'frame {i}', fontsize=10)
    cap.release(); plt.tight_layout(); plt.show()

def add_label(name, bowler, arm, bfc, ffc, br):
    assert arm in ('R', 'L') and bfc < ffc < br, 'arm must be R/L and BFC < FFC < BR'
    _, fps = video_info(name)
    row = dict(clip_id=Path(name).stem, bowler_id=bowler, bowling_arm=arm, fps=round(fps, 3), bfc=bfc, ffc=ffc, br=br)
    df = pd.read_csv(LABELS) if LABELS.exists() else pd.DataFrame(columns=row.keys())
    df = pd.concat([df[df.clip_id != row['clip_id']], pd.DataFrame([row])], ignore_index=True)
    df.to_csv(LABELS, index=False)
    print('saved', row)

print(sorted(p.name for p in VIDEOS.iterdir()))

In [ ]:
# Example – change the file name and ranges for each clip
video_info('clip01.mp4')
show_frames('clip01.mp4', step=5)                  # 1) whole clip, every 5th frame

In [ ]:
show_frames('clip01.mp4', start=40, end=60, step=1, crop_bottom=0.5)   # 2) zoom in on the feet

In [ ]:
add_label('clip01.mp4', bowler='B01', arm='R', bfc=0, ffc=0, br=0)     # 3) put your frame numbers here

In [ ]:
pd.read_csv(LABELS)   # check all labels

## 6. Rule baseline + XGBoost (leave-one-bowler-out)

In [ ]:
!python train_evaluate.py --landmarks "{ROOT}/data/landmarks" --labels "{LABELS}" --out "{ROOT}/results" --save-model "{ROOT}/models/event_xgb.json"

In [ ]:
display(pd.read_csv(f'{ROOT}/results/summary.csv'))
display(pd.read_csv(f'{ROOT}/results/angle_error.csv'))

## 7. Predict events on a new clip

In [ ]:
!python predict.py --landmarks "{ROOT}/data/landmarks/clip01.csv" --arm R --fps 60 --model "{ROOT}/models/event_xgb.json"